# 실전 11-2: MCP 실무 연동 (LangChain Client + MCP Server)

## 1. 실험 목적
- 앞선 노트북에서 만든 '기상청 MCP 서버'를 백그라운드로 실행하고,
- 랭체인으로 짠 우리의 LLM 에이전트(클라이언트)가 이 서버에 접근해서 기상청 데이터를 가져와 사용자에게 답변하도록 만듭니다.

## 2. 서버 실행용 스크립트 작성
실무에서는 MCP 서버가 항상 켜져 있어야 하므로, 파이썬 파일(`.py`)로 따로 빼서 실행합니다.
이 셀을 실행하면 `weather_mcp_server.py` 파일이 현재 폴더에 자동 생성됩니다.

In [4]:
%%writefile weather_mcp_server.py
import asyncio
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

app = Server("Weather_MCP_Server")

@app.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="get_weather",
            description="주어진 도시의 날씨를 알려줍니다. (예: Seoul, Tokyo)",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        )
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    if name == "get_weather":
        city = arguments.get("city")
        fake_db = {"Seoul": "맑음, 25도", "Tokyo": "비, 20도", "New York": "눈, -2도"}
        result = fake_db.get(city, "해당 도시의 날씨를 모릅니다.")
        return [TextContent(type="text", text=result)]
    raise ValueError(f"알 수 없는 도구입니다: {name}")

async def main():
    # 터미널 표준 입출력(stdio)을 통해 MCP 서버를 오픈합니다.
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())


Overwriting weather_mcp_server.py


## 3. LangChain MCP Client 세팅
이제 우리가 만든 에이전트가 백그라운드의 저 `weather_mcp_server.py` 프로세스와 소통(stdio 통신)할 수 있도록 연결(Connect)합니다.

In [5]:
import os
import asyncio
from contextlib import AsyncExitStack
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp.client.session import ClientSession

# LangChain MCP Toolkit (MCP 서버의 툴들을 랭체인 툴로 자동 변환해줌)
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

# 파이썬 명령어 위치 (가상환경 사용 시 해당 파이썬 경로 입력 필요)
import sys
python_exe = sys.executable 

# 서버 파라미터 (명령어: `python weather_mcp_server.py`)
server_params = StdioServerParameters(
    command=python_exe,
    args=["weather_mcp_server.py"],
)

## 4. MCP 서버에 접속하고 LLM 구동하기
연결이 맺어지면, 서버에 있는 `get_weather` 도구를 가져와서 LLM에게 넘겨줍니다.

In [ ]:
async def run_mcp_agent():
    async with AsyncExitStack() as stack:
        # 1. MCP 서버와 통신 파이프라인(stdio) 열기
        read, write = await stack.enter_async_context(stdio_client(server_params))
        
        # 2. 클라이언트 세션 시작
        session = await stack.enter_async_context(ClientSession(read, write))
        await session.initialize()
        
        # 3. 마법이 일어나는 순간! MCP 서버에 등록된 모든 툴을 LangChain 툴로 쓱싹 가져옴
        tools = await load_mcp_tools(session)
        print(f"✅ MCP 서버에서 {len(tools)}개의 도구를 가져왔습니다: {[t.name for t in tools]}")

        # 4. 에이전트 생성 (이 LLM은 이제 MCP 서버의 툴을 맘대로 씁니다)
        agent = create_agent(
            model=llm, 
            tools=tools, 
            system_prompt="당신은 유능한 비서입니다. 주어진 도구를 사용하여 날씨를 확인하세요."
        )

        # 5. 질문하기
        question = "도쿄(Tokyo)와 서울(Seoul) 날씨 어때?"
        print(f"\n[사용자 질문]: {question}")
        
        result = await agent.ainvoke({"messages": [("user", question)]})
        print(f"\n[에이전트 답변]:\n{result['messages'][-1].content}")

# 실행
await run_mcp_agent()

## 5. 결과 해석 및 의의
- 랭체인 안에는 `get_weather`라는 파이썬 함수가 전혀 없었습니다!
- 랭체인 에이전트가 MCP 프로토콜을 통해, 로컬에 떠있는 `weather_mcp_server.py` 프로세스에게 "도쿄 날씨 내놔" 라고 명령을 내렸고, 서버가 결과를 뱉어준 것입니다.
- 실무에서는 저 서버가 **사내 오라클 데이터베이스 서버**이거나 **Notion 워크스페이스**가 됩니다. 우리는 그저 `StdioServerParameters`의 URL만 바꿔끼우면, 수만 개의 사내 데이터를 이 에이전트가 그대로 다룰 수 있게 되는 엄청난 확장성을 가지게 됩니다.